# Clean chess datasets

Run all cells to clean every `.txt` source under `dataset/raw/`. The sources are unchanged. Results go to `dataset/processed/`: `cleaned.txt`, `provenance.jsonl`, and `report.json`. No extra Python packages are needed by the cleaning logic.

In [ ]:
from pathlib import Path
import sys

project_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'scripts' / 'clean_dataset.py').is_file()
                     and (path / 'dataset').is_dir()), None)
if project_root is None:
    raise FileNotFoundError('Open this notebook from the MyTorch project or notebooks/ folder')
sys.path.insert(0, str(project_root))
from scripts.clean_dataset import clean_dataset

dataset_dir = project_root / 'dataset'
source_dir = dataset_dir / 'raw' if (dataset_dir / 'raw').is_dir() else dataset_dir
print('Source:', source_dir)

In [ ]:
report = clean_dataset(dataset_dir)
print('Cleaned data:', dataset_dir / 'processed' / 'cleaned.txt')
print('Report:', dataset_dir / 'processed' / 'report.json')

## Summary

A conflict means identical model inputs have different class labels. Every record in that group is excluded.

In [ ]:
print(f"Records: {report['records']:,} | Retained: {report['accepted']:,} | "
      f"Duplicates: {report['duplicates']:,} | Rejected: {len(report['rejected']):,} | "
      f"Conflicting records: {report['conflicting_records']:,}")

largest = max(report['by_class'].values(), default=1) or 1
for label, count in report['by_class'].items():
    bar = '█' * round(30 * count / largest)
    print(f'{label:<10} {count:>10,}  {bar}')

## Sources and issues

`Retained` counts distinct output positions originating from each source. A position found in two source files appears in both counts. The complete references for rejected and conflicting records are in `report.json`.

In [ ]:
print(f"{'Source':<44} {'Records':>10} {'Retained':>10} {'Rejected':>10}")
for source, counts in report['by_source'].items():
    print(f"{source:<44} {counts['records']:>10,} {counts['retained']:>10,} {counts['rejected']:>10,}")

for item in report['rejected'][:20]:
    print(f"Rejected {item['file']}:{item['line']}: {item['reason']}")
for group in report['conflicts'][:20]:
    print('Conflict:', ', '.join(f"{ref['file']}:{ref['line']} ({ref['label']})"
                                 for ref in group['sources']))
if len(report['rejected']) > 20 or len(report['conflicts']) > 20:
    print('Additional details: dataset/processed/report.json')